# Modelo-07 — Construção da Tabela Mestra do Sistema PSD

## Objetivo

Este notebook tem como objetivo integrar e padronizar os dados históricos do sistema de Portas de Plataforma (PSD), preparando uma tabela mestra para as etapas posteriores de processamento e treinamento do `CatBoostClassifier`.

A unidade de análise será o sistema PSD de cada **estação e plataforma** em uma determinada data. Dessa forma, os alarmes e as manutenções dos diferentes componentes — como PDMs, MDUs, sensores, controladores e portas de emergência — poderão representar conjuntamente o estado operacional daquele sistema.

Neste notebook serão realizadas somente a leitura, integração, padronização e auditoria das fontes. A criação das features e do alvo — ocorrência de uma falha nos 7 dias seguintes — será feita no notebook de processamento, evitando misturar a preparação dos eventos com a modelagem.

## Hipóteses que orientam a construção da tabela

A tabela mestra deve preservar os campos necessários para testar e transformar as evidências levantadas na Sprint 1 em features temporais. A inclusão de uma informação na tabela não significa que ela será obrigatoriamente utilizada pelo modelo.

| Hipótese | Resultado da investigação | Informação que deve ser preservada |
|---|---|---|
| H1 — padrões por horário | Confirmada | Horário do alarme, duração e identificação do PSD |
| H2 — regime semanal | Confirmada | Data do evento para derivar dia da semana e domingo |
| H3 — fluxo de passageiros | Refutada | Dados de catraca não serão integrados na primeira versão |
| H4 — aumento de alarmes antes da corretiva | Não confirmada | Tipo e horário dos alarmes serão preservados, sem assumir que a contagem sempre cresce antes da falha |
| H5 — tempo de normalização | Possui sinal, mas é insuficiente isoladamente | Início, normalização, duração e estação |
| H6 — posição central ou extremidade | Efeito prático pequeno | A posição da porta não será uma feature prioritária no modelo agregado por plataforma |
| H7 — predominância de falhas mecânicas | Sustentada | Causa, defeito, objeto e ação das manutenções históricas |

> **Prevenção de vazamento temporal:** causa, defeito e ação só poderão gerar features depois que a respectiva informação estiver disponível no histórico. Esses campos nunca poderão descrever a própria falha que o modelo está tentando prever.

In [2]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semente: {RANDOM_SEED}")

Pandas: 3.0.5
NumPy: 2.5.1
Semente: 42


In [5]:
df_alarmes = pd.read_csv("../../data/Alarmes/alarmes_trata_limpos_consolidado.csv")
df_corretiva_emergencial = pd.read_excel("../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx")
df_corretiva_programada = pd.read_excel("../../data/Corretiva Programada/processados/ZM02_limpa.xlsx")
df_preventiva = pd.read_csv("../../data/Preventiva/preventiva_zm03_limpa.csv")

print("Alarmes:", df_alarmes.shape)
print("Corretivas emergenciais:", df_corretiva_emergencial.shape)
print("Corretivas programadas:", df_corretiva_programada.shape)
print("Preventivas:", df_preventiva.shape)

Alarmes: (742506, 15)
Corretivas emergenciais: (10543, 27)
Corretivas programadas: (1163, 27)
Preventivas: (569, 19)


In [6]:
bases = {
    "alarmes": df_alarmes,
    "corretiva_emergencial": df_corretiva_emergencial,
    "corretiva_programada": df_corretiva_programada,
    "preventiva": df_preventiva,
}

for nome, df in bases.items():
    print(f"\n{nome.upper()} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")

    resumo_colunas = pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "nulos_%": (df.isna().mean().values * 100).round(2),
    })

    display(resumo_colunas)
    display(df.head(3))


ALARMES — 742,506 linhas × 15 colunas


,coluna,tipo,nulos,nulos_%
0,TAG,str,0,0.00
1,Horário do Alarme,str,0,0.00
2,Estação,str,0,0.00
3,Porta,float64,170,0.02
4,Alarme,str,0,0.00
5,Horário da Normalizacao,str,0,0.00
6,Tempo de Normalização,str,0,0.00
7,tempo_normalizacao_seg,int64,0,0.00
8,dt_alarme,str,0,0.00
9,dt_normalizacao,str,0,0.00


,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização,tempo_normalizacao_seg,dt_alarme,dt_normalizacao,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido
0,PSD_FAL_FAL2_PDM09_OBS,2026-05-12 08:16:52.657,FAL,9.00,OBS,2026-05-12 08:16:53.670,"0 dias, 00:00:01",1,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670,1.01,FAL,FAL2,9.00,OBS
1,PSD_LUZ_LUZ1_PDM08_OBS,2026-05-12 08:15:09.913,LUZ,8.00,OBS,2026-05-12 08:15:10.917,"0 dias, 00:00:01",1,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917,1.00,LUZ,LUZ1,8.00,OBS
2,PSD_FAL_FAL1_PDM15_OBS,2026-05-12 08:14:06.657,FAL,15.00,OBS,2026-05-12 08:14:07.670,"0 dias, 00:00:01",1,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670,1.01,FAL,FAL1,15.00,OBS



CORRETIVA_EMERGENCIAL — 10,543 linhas × 27 colunas


,coluna,tipo,nulos,nulos_%
0,linha_origem,int64,0,0.00
1,tipo_de_nota,str,0,0.00
2,nota,int64,0,0.00
3,ordem,float64,1,0.01
4,perfil_do_catalogo,str,0,0.00
5,descricao,str,0,0.00
6,local_de_instalacao,str,0,0.00
7,centro_de_trabalho_responsavel,str,0,0.00
8,inicio_avaria_datetime,datetime64[us],0,0.00
9,fim_avaria_datetime,datetime64[us],326,3.09


...

,coluna,tipo,nulos,nulos_%
0,linha_origem,int64,0,0.00
1,tipo_de_nota,str,0,0.00
2,nota,int64,0,0.00
3,ordem,float64,5,0.43
4,perfil_do_catalogo,str,0,0.00
5,descricao,str,0,0.00
6,local_de_instalacao,str,0,0.00
7,centro_de_trabalho_responsavel,str,0,0.00
8,inicio_avaria_datetime,datetime64[us],0,0.00
9,fim_avaria_datetime,datetime64[us],39,3.35


,linha_origem,tipo_de_nota,nota,ordem,perfil_do_catalogo,descricao,local_de_instalacao,centro_de_trabalho_responsavel,inicio_avaria_datetime,fim_avaria_datetime,duracao_avaria_horas,encerramento_datetime,indisponivel_flag,analisada_flag,ordem_ausente_flag,ordem_repetida_flag,nota_repetida_flag,duracao_extrema_flag,duracao_zero_flag,status_do_usuario_da_ordem,texto_longo_da_ordem,status_do_sistema_da_ordem,sintoma,objeto,defeito,causa,acao
0,2,Z2,15069252,"54,081,268.00",6PSD,Porta PCC-PSD a,SP04-6-PSD-BUT,SP04IBUT,2026-01-25 08:20:00,2026-01-25 09:30:00,1.17,2026-01-25 16:50:18,False,False,False,False,False,False,False,LIBO EXDO,Porta PCC-PSD 25.01.2026 08:29:08 BRAZIL (28-01325) 25.01.2026 16:43:35 BRAZIL (28-04512) Realizado a instalação de ...,ENTE CNPA CAPC NOLQ SCDM,PDM NÃO ABRE,"ESTRUTURA, ACABAMENTO, E ACESSÓRIOS DO P",PORTA DESAJUSTADA,FALTA FIXACAO,FIXADO
1,3,Z2,14647619,"53,659,484.00",6PSD,PDMs plat 1 e 2 com falha momentânea.,SP04-6-PSD-BUT-P1S1,SP04IBUT,2025-05-30 09:30:00,2025-06-01 04:00:00,42.50,2025-06-01 04:36:44,False,False,False,False,False,False,False,LIBO EXDO,PDMs plat 1 e 2 com falha momentânea. 30.05.2025 14:24:19 BRAZIL DENIS RODRIGUES DO NASCIMENTO (28-04212) O CCO info...,ENTE CNPA CAPC NOLQ SCDM,MAU CONTATO,SWITCK DE COMUNICAÇÃO,SEM CONEXAO ELETRICA,ERRO DE SOFTWARE,RESETADO E TESTADO
2,4,Z2,14682623,"53,694,340.00",6PSD,Rele painel SSE intermitente,SP04-6-PSD-BUT-P2S1-PSSE2,SP04IBUT,2025-06-19 02:00:00,2025-06-19 04:30:00,2.50,2025-06-19 05:10:12,False,False,False,False,False,False,False,LIBO EXDO,Rele painel SSE intermitente 19.06.2025 05:05:07 BRAZIL D. DE MELO SOARES (28-00570) Tel. 55 11 458.. Realizado a su...,ENTE CNPA CAPC NOLQ SCDM,FACHADA PSD SEM ALIMENTAÇÃO,DISPOSITIVO DE PROTECAO,CONVERSOR DE TENSAO AVARIADO,OPERACAO INDEVIDA,SUBSTITUIDO(A)



PREVENTIVA — 569 linhas × 19 colunas


,coluna,tipo,nulos,nulos_%
0,Ordem,int64,0,0.00
1,Tipo de ordem,str,0,0.00
2,Centro de trabalho,str,0,0.00
3,Local de instalação,str,0,0.00
4,Texto breve,str,0,0.00
5,Plano de manutenção,int64,0,0.00
6,Status usuário,str,0,0.00
7,Status do sistema,str,0,0.00
8,Data-base do início,str,0,0.00
9,Hr.início programada,str,0,0.00


,Ordem,Tipo de ordem,Centro de trabalho,Local de instalação,Texto breve,Plano de manutenção,Status usuário,Status do sistema,Data-base do início,Hr.início programada,Data-base do fim,Fim-base (hora),DT_Inicio,DT_Fim,duracao_h,Estacao_Centro,Estacao_Local,Estacao,cancelada
0,54164639,ZM03,SP04IPTA,SP04-6-PSD-PTA,PSD - RACK DO PCC (AR COND.) PTA P1,47029,LIBO EXDO ANEX,ENTE CNPA CAPC NOLQ SCDM,2026-03-23,21:30:00,2026-03-24,04:30:00,2026-03-23 21:30:00,2026-03-24 04:30:00,7.00,PTA,PTA,PTA,False
1,54164638,ZM03,SP04IFAL,SP04-6-PSD-FAL,PSD - RACK DO PCC (AR COND.) FAL P1,47028,LIBO EXDO,ENTE CNPA CAPC MatC NOLQ,2026-03-24,21:30:00,2026-03-25,04:30:00,2026-03-24 21:30:00,2026-03-25 04:30:00,7.00,FAL,FAL,FAL,False
2,54152224,ZM03,SP04IPIN,SP04-6-PSD-PIN,PSD - RACK DO PCC (AR COND.) PIN P1,47027,LIBO EXDO ANEX,ENTE CNPA CAPC NOLQ SCDM,2026-03-17,21:30:00,2026-03-18,04:30:00,2026-03-17 21:30:00,2026-03-18 04:30:00,7.00,PIN,PIN,PIN,False


In [7]:
import unicodedata

def padronizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", str(nome).strip().lower())
    nome = nome.encode("ascii", errors="ignore").decode("utf-8")
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")

for nome_base, df in bases.items():
    colunas_originais = df.columns.tolist()
    colunas_padronizadas = [padronizar_nome_coluna(coluna) for coluna in colunas_originais]

    if len(colunas_padronizadas) != len(set(colunas_padronizadas)):
        raise ValueError(f"A padronização gerou colunas duplicadas na base {nome_base}.")

    df.columns = colunas_padronizadas
    print(f"{nome_base}: {df.columns.tolist()}")

alarmes: ['tag', 'horario_do_alarme', 'estacao', 'porta', 'alarme', 'horario_da_normalizacao', 'tempo_de_normalizacao', 'tempo_normalizacao_seg', 'dt_alarme', 'dt_normalizacao', 'duracao_calculada_seg', 'estacao_extraida', 'plataforma_extraida', 'porta_extraida', 'tipo_evento_extraido']
corretiva_emergencial: ['linha_origem', 'tipo_de_nota', 'nota', 'ordem', 'perfil_do_catalogo', 'descricao', 'local_de_instalacao', 'centro_de_trabalho_responsavel', 'inicio_avaria_datetime', 'fim_avaria_datetime', 'duracao_avaria_horas', 'analise_datetime', 'encerramento_datetime', 'indisponivel_flag', 'falha_flag', 'analisada_flag', 'ordem_ausente_flag', 'ordem_repetida_flag', 'comentarios_classificacao_engenharia', 'status_do_usuario_da_ordem', 'texto_longo_da_ordem', 'status_do_sistema_da_ordem', 'sintoma', 'objeto', 'defeito', 'causa', 'acao']
corretiva_programada: ['linha_origem', 'tipo_de_nota', 'nota', 'ordem', 'perfil_do_catalogo', 'descricao', 'local_de_instalacao', 'centro_de_trabalho_responsa